In [1]:
import pandas as pd
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import math
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [2]:
cadrms_latest = pd.read_excel('FireCADRMS_inci_types_latest.xlsx', parse_dates=['Received_Datetime'], index_col='Received_Datetime')

In [3]:
cadrms_latest.head(10)

,unique_Id,inci_no,inci_type,incident_category,Incident_Type_Description,Property_Loss_Value,Content_Loss_Value,Property_Value,Content_Value,Civilian_Fatal,Civilian_Injuries,prop_use,Property_Damage_Category,Property_Damage_Description,Dispatched_Datetime,Arrival_Datetime,Cleared_Datetime,DAUID,Longitude,Latitude
Received_Datetime,,,,,,,,,,,,,,,,,,,,
2020-10-31 05:21:00,397231,20-0006971,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,355,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-10-31 05:21:00,2020-10-31 05:27:00,2020-10-31 05:33:00,35431013.0,-79.697299,44.364071
2020-11-02 17:48:00,397468,20-0007034,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,323,320,Multi-Unit Dwelling - Over 12 Units ...,2020-11-02 17:48:00,2020-11-02 17:53:00,2020-11-02 18:22:00,35431321.0,-79.699290,44.354853
2020-11-03 15:16:00,397549,20-0007052,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,355,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-03 15:16:00,2020-11-03 15:22:00,2020-11-03 15:39:00,35431013.0,-79.697299,44.364071
2020-11-09 02:53:00,398004,20-0007183,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,323,320,Multi-Unit Dwelling - Over 12 Units ...,2020-11-09 02:53:00,2020-11-09 02:59:00,2020-11-09 03:06:00,35431321.0,-79.699290,44.354853
2020-11-11 00:38:00,398196,20-0007238,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,355,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-11 00:38:00,2020-11-11 00:45:00,2020-11-11 00:51:00,35431013.0,-79.697299,44.364071
2020-11-20 11:22:00,399163,20-0007472,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,896,890,"Sidewalk, street, roadway (do not use for fire...",2020-11-20 11:21:00,2020-11-20 11:27:00,2020-11-20 11:28:00,35430688.0,-79.704340,44.346868
2020-11-22 14:26:00,399317,20-0007519,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,355,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-22 14:26:00,2020-11-22 14:32:00,2020-11-22 14:37:00,35431013.0,-79.697299,44.364071
2020-11-25 21:18:00,399595,20-0007595,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,355,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-25 21:18:00,2020-11-25 21:24:00,2020-11-25 21:27:00,35431013.0,-79.697299,44.364071
2020-11-29 11:57:00,399831,20-0007669,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,0,355,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-29 11:57:00,2020-11-29 12:03:00,2020-11-29 12:21:00,35431013.0,-79.697299,44.364071


In [4]:
cadrms_features = cadrms_latest[['incident_category', 'Property_Damage_Category', 'DAUID', 'Longitude', 'Latitude']]

In [5]:
cadrms_features

,incident_category,Property_Damage_Category,DAUID,Longitude,Latitude
Received_Datetime,,,,,
2020-10-31 05:21:00,I,350,35431013.0,-79.697299,44.364071
2020-11-02 17:48:00,I,320,35431321.0,-79.699290,44.354853
2020-11-03 15:16:00,I,350,35431013.0,-79.697299,44.364071
2020-11-09 02:53:00,I,320,35431321.0,-79.699290,44.354853
2020-11-11 00:38:00,I,350,35431013.0,-79.697299,44.364071
...,...,...,...,...,...
2023-12-17 01:26:00,A,900,NaN,-79.698410,44.422628
2023-09-05 05:59:00,H,900,NaN,-79.653430,44.416032
2021-08-04 13:20:00,J,900,NaN,-79.646030,44.415317


In [6]:
cadrms_features['incident_category'].value_counts()

incident_category
I                            28394
E                             3814
H                             2809
J                             1424
G                             1131
D                              985
F                              820
A                              783
C                              649
B                                5
Name: count, dtype: int64

In [7]:
cadrms_features.dtypes

incident_category            object
Property_Damage_Category      int64
DAUID                       float64
Longitude                   float64
Latitude                    float64
dtype: object

In [8]:
le = LabelEncoder()
cadrms_features['incident_category'] = le.fit_transform(cadrms_features['incident_category'])

In [9]:
cadrms_features['incident_category'].value_counts()

incident_category
8    28394
4     3814
7     2809
9     1424
6     1131
3      985
5      820
0      783
2      649
1        5
Name: count, dtype: int64

In [10]:
cadrms_features.isna().sum()

incident_category             0
Property_Damage_Category      0
DAUID                       245
Longitude                     0
Latitude                      0
dtype: int64

In [23]:
cadrms_features_cleaned = cadrms_features.dropna()

In [24]:
cadrms_features_cleaned.isna().sum()

incident_category           0
Property_Damage_Category    0
DAUID                       0
Longitude                   0
Latitude                    0
dtype: int64

In [25]:
cadrms_features_cleaned['DAUID'] = cadrms_features_cleaned['DAUID'].astype(int)

In [26]:
cadrms_features_cleaned.dtypes

incident_category             int32
Property_Damage_Category      int64
DAUID                         int32
Longitude                   float64
Latitude                    float64
dtype: object

### The dataset is now clean with correct datatypes and no missing values 

In [27]:
cadrms_features_cleaned

,incident_category,Property_Damage_Category,DAUID,Longitude,Latitude
Received_Datetime,,,,,
2020-10-31 05:21:00,8,350,35431013,-79.697299,44.364071
2020-11-02 17:48:00,8,320,35431321,-79.699290,44.354853
2020-11-03 15:16:00,8,350,35431013,-79.697299,44.364071
2020-11-09 02:53:00,8,320,35431321,-79.699290,44.354853
2020-11-11 00:38:00,8,350,35431013,-79.697299,44.364071
...,...,...,...,...,...
2022-08-23 10:13:00,0,900,35431380,-79.683722,44.308802
2022-10-18 14:36:00,0,900,35431044,-79.688199,44.410711
2023-04-10 14:11:00,0,900,35431324,-79.693849,44.359211


In [28]:
#Resampling the time-series data and obtain statistics for an hourly interval
cadrms_features_cleaned = cadrms_features_cleaned.resample('1H').agg({'incident_category': lambda x: stats.mode(x)[0],
                                                                      'Property_Damage_Category': lambda x: stats.mode(x)[0],
                                                                      'DAUID': lambda x: stats.mode(x)[0],
                                                                      'Longitude': lambda x: stats.mode(x)[0],
                                                                      'Latitude': lambda x: stats.mode(x)[0]})

In [29]:
cadrms_features_cleaned

,incident_category,Property_Damage_Category,DAUID,Longitude,Latitude
Received_Datetime,,,,,
2020-01-01 00:00:00,8.0,320.0,35430669.0,-79.725625,44.324085
2020-01-01 01:00:00,8.0,500.0,35431060.0,-79.691135,44.390391
2020-01-01 02:00:00,0.0,350.0,35431015.0,-79.709545,44.373663
2020-01-01 03:00:00,8.0,500.0,35430707.0,-79.729339,44.325454
2020-01-01 04:00:00,8.0,220.0,35430996.0,-79.693589,44.358741
...,...,...,...,...,...
2023-12-31 19:00:00,NaN,NaN,NaN,NaN,NaN
2023-12-31 20:00:00,0.0,860.0,35431058.0,-79.705895,44.400154
2023-12-31 21:00:00,8.0,300.0,35431032.0,-79.715016,44.397739


In [30]:
cadrms_features_cleaned.isna().sum()

incident_category           12088
Property_Damage_Category    12088
DAUID                       12088
Longitude                   12088
Latitude                    12088
dtype: int64

In [31]:
cadrms_features_cleaned = cadrms_features_cleaned.dropna()

In [32]:
cadrms_features_cleaned

,incident_category,Property_Damage_Category,DAUID,Longitude,Latitude
Received_Datetime,,,,,
2020-01-01 00:00:00,8.0,320.0,35430669.0,-79.725625,44.324085
2020-01-01 01:00:00,8.0,500.0,35431060.0,-79.691135,44.390391
2020-01-01 02:00:00,0.0,350.0,35431015.0,-79.709545,44.373663
2020-01-01 03:00:00,8.0,500.0,35430707.0,-79.729339,44.325454
2020-01-01 04:00:00,8.0,220.0,35430996.0,-79.693589,44.358741
...,...,...,...,...,...
2023-12-31 17:00:00,8.0,300.0,35431054.0,-79.721314,44.385029
2023-12-31 18:00:00,8.0,220.0,35431059.0,-79.693589,44.391539
2023-12-31 20:00:00,0.0,860.0,35431058.0,-79.705895,44.400154


### Check whether the data is stationary

In [33]:
for i in range(len(cadrms_features_cleaned.columns)):
    adfuller_result = adfuller(cadrms_features_cleaned[cadrms_features_cleaned.columns[i]])
    if adfuller_result[1] > 0.05:
        print('{} = Series is not stationary'.format(cadrms_features_cleaned.columns[i]))
    else:
        print('{} = Series is stationary'.format(cadrms_features_cleaned.columns[i]))

incident_category = Series is stationary
Property_Damage_Category = Series is stationary
DAUID = Series is stationary
Longitude = Series is stationary
Latitude = Series is stationary


### The data is stationary

### Run Granger Causality Test to check whether one variable causes the other

### Null Hypothesis: Incident Category does not granger cause dissemination area.
### Alternate Hypothesis: Incident Category granger causes dissemination area.

In [34]:
max_lags = 8
print('Does incident category cause fire incident in the dissemination area?')
granger_result1 = grangercausalitytests(cadrms_features_cleaned[['DAUID', 'incident_category']], max_lags)

Does incident category cause fire incident in the dissemination area?

Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.1497  , p=0.6988  , df_denom=22972, df_num=1
ssr based chi2 test:   chi2=0.1498  , p=0.6988  , df=1
likelihood ratio test: chi2=0.1498  , p=0.6988  , df=1
parameter F test:         F=0.1497  , p=0.6988  , df_denom=22972, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=1.3675  , p=0.2548  , df_denom=22969, df_num=2
ssr based chi2 test:   chi2=2.7357  , p=0.2547  , df=2
likelihood ratio test: chi2=2.7355  , p=0.2547  , df=2
parameter F test:         F=1.3675  , p=0.2548  , df_denom=22969, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=0.9677  , p=0.4068  , df_denom=22966, df_num=3
ssr based chi2 test:   chi2=2.9039  , p=0.4067  , df=3
likelihood ratio test: chi2=2.9038  , p=0.4067  , df=3
parameter F test:         F=0.9677  , p=0.4068  , df_denom=22966, df_num=3

Granger Cau

### Since p>0.05 in all the lags, we fail to reject the null hypothesis 

In [35]:
print('Does property damage category cause fire incident in the dissemination area?')
granger_result2 = grangercausalitytests(cadrms_features_cleaned[['DAUID', 'Property_Damage_Category']], max_lags)

Does property damage category cause fire incident in the dissemination area?

Granger Causality
number of lags (no zero) 1
ssr based F test:         F=1.2561  , p=0.2624  , df_denom=22972, df_num=1
ssr based chi2 test:   chi2=1.2562  , p=0.2624  , df=1
likelihood ratio test: chi2=1.2562  , p=0.2624  , df=1
parameter F test:         F=1.2561  , p=0.2624  , df_denom=22972, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=2.1155  , p=0.1206  , df_denom=22969, df_num=2
ssr based chi2 test:   chi2=4.2319  , p=0.1205  , df=2
likelihood ratio test: chi2=4.2315  , p=0.1205  , df=2
parameter F test:         F=2.1155  , p=0.1206  , df_denom=22969, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=1.6652  , p=0.1721  , df_denom=22966, df_num=3
ssr based chi2 test:   chi2=4.9971  , p=0.1720  , df=3
likelihood ratio test: chi2=4.9966  , p=0.1720  , df=3
parameter F test:         F=1.6652  , p=0.1721  , df_denom=22966, df_num=3

Gran

### Till lag 7, p>0.05 and for lag 8, p<0.05 so for most of the lags we reject H0 

In [36]:
print('Does longitude cause fire incident in the dissemination area?')
granger_result3 = grangercausalitytests(cadrms_features_cleaned[['DAUID', 'Longitude']], max_lags)

Does longitude cause fire incident in the dissemination area?

Granger Causality
number of lags (no zero) 1
ssr based F test:         F=2.9032  , p=0.0884  , df_denom=22972, df_num=1
ssr based chi2 test:   chi2=2.9036  , p=0.0884  , df=1
likelihood ratio test: chi2=2.9034  , p=0.0884  , df=1
parameter F test:         F=2.9032  , p=0.0884  , df_denom=22972, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=4.2435  , p=0.0144  , df_denom=22969, df_num=2
ssr based chi2 test:   chi2=8.4888  , p=0.0143  , df=2
likelihood ratio test: chi2=8.4873  , p=0.0144  , df=2
parameter F test:         F=4.2435  , p=0.0144  , df_denom=22969, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=2.8964  , p=0.0337  , df_denom=22966, df_num=3
ssr based chi2 test:   chi2=8.6918  , p=0.0337  , df=3
likelihood ratio test: chi2=8.6901  , p=0.0337  , df=3
parameter F test:         F=2.8964  , p=0.0337  , df_denom=22966, df_num=3

Granger Causality
n

### For the first lag, p>0.05 whereas for second lag p<0.05 which means we reject the null hypothesis.

In [37]:
print('Does latitude cause fire incident in the dissemination area?')
granger_result4 = grangercausalitytests(cadrms_features_cleaned[['DAUID', 'Latitude']], max_lags)

Does latitude cause fire incident in the dissemination area?

Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.0385  , p=0.8444  , df_denom=22972, df_num=1
ssr based chi2 test:   chi2=0.0385  , p=0.8444  , df=1
likelihood ratio test: chi2=0.0385  , p=0.8444  , df=1
parameter F test:         F=0.0385  , p=0.8444  , df_denom=22972, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=0.2594  , p=0.7715  , df_denom=22969, df_num=2
ssr based chi2 test:   chi2=0.5189  , p=0.7715  , df=2
likelihood ratio test: chi2=0.5189  , p=0.7715  , df=2
parameter F test:         F=0.2594  , p=0.7715  , df_denom=22969, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=0.3878  , p=0.7618  , df_denom=22966, df_num=3
ssr based chi2 test:   chi2=1.1638  , p=0.7617  , df=3
likelihood ratio test: chi2=1.1638  , p=0.7617  , df=3
parameter F test:         F=0.3878  , p=0.7618  , df_denom=22966, df_num=3

Granger Causality
nu

### For all the lags, p>0.05 so we fail to reject H0

### Features are selected based on lowest p-value rule where lowest p-value is less than the threshold(0.05) across lags is considered for each feature.
### Since, Property_Damage_Category and Longitude has the lowest p-value less than the threshold, those features along with DAUID are selected.

In [38]:
cadrms_features_cleaned = cadrms_features_cleaned[['Property_Damage_Category', 'DAUID', 'Longitude']]

In [39]:
cadrms_features_cleaned

,Property_Damage_Category,DAUID,Longitude
Received_Datetime,,,
2020-01-01 00:00:00,320.0,35430669.0,-79.725625
2020-01-01 01:00:00,500.0,35431060.0,-79.691135
2020-01-01 02:00:00,350.0,35431015.0,-79.709545
2020-01-01 03:00:00,500.0,35430707.0,-79.729339
2020-01-01 04:00:00,220.0,35430996.0,-79.693589
...,...,...,...
2023-12-31 17:00:00,300.0,35431054.0,-79.721314
2023-12-31 18:00:00,220.0,35431059.0,-79.693589
2023-12-31 20:00:00,860.0,35431058.0,-79.705895


In [40]:
#training/testing data split is 80% and 20%
X_train = cadrms_features_cleaned[:int(0.8*(len(cadrms_features_cleaned)))]
X_test = cadrms_features_cleaned[int(0.8*(len(cadrms_features_cleaned))):]

In [41]:
X_train.shape

(18380, 3)

In [42]:
X_test.shape

(4596, 3)

In [43]:
var_model = VAR(X_train)

In [44]:
#Find the lag that is suitable for the model
lag_order = var_model.select_order(maxlags=20)
print(lag_order.summary())

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       13.32*      13.32*  6.110e+05*      13.32*
1        13.32       13.33   6.111e+05       13.32
2        13.32       13.33   6.110e+05       13.33
3        13.32       13.34   6.112e+05       13.33
4        13.32       13.34   6.115e+05       13.33
5        13.32       13.34   6.117e+05       13.33
6        13.32       13.35   6.120e+05       13.33
7        13.32       13.35   6.121e+05       13.33
8        13.32       13.36   6.123e+05       13.34
9        13.32       13.36   6.122e+05       13.34
10       13.33       13.37   6.126e+05       13.34
11       13.33       13.37   6.126e+05       13.34
12       13.33       13.37   6.130e+05       13.34
13       13.33       13.38   6.132e+05       13.34
14       13.33       13.38   6.137e+05       13.35
15       13.33       13.39   6.136e+05       13.35
16       13.33       13.39   6.

In [45]:
#Fitting the model using the best lag where AIC, BIC, FPE and HQIC are minimum
var_model1 = VAR(X_train)
result1 = var_model1.fit(1)

In [46]:
print(result1.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 18, Jun, 2024
Time:                     17:45:58
--------------------------------------------------------------------
No. of Equations:         3.00000    BIC:                    13.3279
Nobs:                     18379.0    HQIC:                   13.3245
Log likelihood:          -200654.    FPE:                    610989.
AIC:                      13.3228    Det(Omega_mle):         610591.
--------------------------------------------------------------------
Results for equation Property_Damage_Category
                                 coefficient       std. error           t-stat            prob
----------------------------------------------------------------------------------------------
const                          236084.543726    320037.384588            0.738           0.461
L1.Property_Damage_Category         0.020724         0.007428            2.790  

In [47]:
var_model2 = VAR(X_train)
result2 = var_model2.fit(2)

In [48]:
print(result2.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 18, Jun, 2024
Time:                     17:46:04
--------------------------------------------------------------------
No. of Equations:         3.00000    BIC:                    13.3319
Nobs:                     18378.0    HQIC:                   13.3259
Log likelihood:          -200635.    FPE:                    611057.
AIC:                      13.3229    Det(Omega_mle):         610360.
--------------------------------------------------------------------
Results for equation Property_Damage_Category
                                 coefficient       std. error           t-stat            prob
----------------------------------------------------------------------------------------------
const                          249681.233205    450958.767206            0.554           0.580
L1.Property_Damage_Category         0.020396         0.007429            2.745  

In [49]:
lag = result1.k_ar

In [50]:
lag

1

In [51]:
#Forecasting the next 10 results using the best lag value calculated previously
prediction1 = result1.forecast(X_train.values[-lag:], steps=10)

In [52]:
prediction1_df = pd.DataFrame(prediction1, index=X_test[0:10].index, columns=cadrms_features_cleaned.columns)

In [53]:
X_test[0:10]

,Property_Damage_Category,DAUID,Longitude
Received_Datetime,,,
2023-04-04 12:00:00,300.0,35430678.0,-79.733311
2023-04-04 13:00:00,320.0,35430690.0,-79.691389
2023-04-04 14:00:00,140.0,35431050.0,-79.691389
2023-04-04 15:00:00,500.0,35431048.0,-79.707972
2023-04-04 18:00:00,300.0,35431022.0,-79.709545
2023-04-04 19:00:00,300.0,35431009.0,-79.712060
2023-04-04 20:00:00,300.0,35430669.0,-79.708608
2023-04-04 21:00:00,150.0,35431060.0,-79.688440
2023-04-04 22:00:00,300.0,35430666.0,-79.732749


In [55]:
prediction1_df

,Property_Damage_Category,DAUID,Longitude
Received_Datetime,,,
2023-04-04 12:00:00,357.222602,3.543100e+07,-79.697067
2023-04-04 13:00:00,358.624004,3.543100e+07,-79.697000
2023-04-04 14:00:00,358.650036,3.543100e+07,-79.697000
2023-04-04 15:00:00,358.650486,3.543100e+07,-79.697000
2023-04-04 18:00:00,358.650494,3.543100e+07,-79.697000
2023-04-04 19:00:00,358.650494,3.543100e+07,-79.697000
2023-04-04 20:00:00,358.650494,3.543100e+07,-79.697000
2023-04-04 21:00:00,358.650494,3.543100e+07,-79.697000
2023-04-04 22:00:00,358.650494,3.543100e+07,-79.697000


In [56]:
lag = result2.k_ar

In [59]:
prediction2 = result2.forecast(X_train.values[-lag:], steps=10)

In [60]:
prediction2_df = pd.DataFrame(prediction2, index=X_test[0:10].index, columns=cadrms_features_cleaned.columns)

In [61]:
prediction2_df

,Property_Damage_Category,DAUID,Longitude
Received_Datetime,,,
2023-04-04 12:00:00,356.149664,3.543100e+07,-79.697502
2023-04-04 13:00:00,357.588372,3.543100e+07,-79.697159
2023-04-04 14:00:00,358.577107,3.543100e+07,-79.697010
2023-04-04 15:00:00,358.623094,3.543100e+07,-79.697004
2023-04-04 18:00:00,358.641020,3.543100e+07,-79.697001
2023-04-04 19:00:00,358.642142,3.543100e+07,-79.697001
2023-04-04 20:00:00,358.642474,3.543100e+07,-79.697000
2023-04-04 21:00:00,358.642499,3.543100e+07,-79.697000
2023-04-04 22:00:00,358.642505,3.543100e+07,-79.697000


In [62]:
#Calculating the Root Mean Square Error(RMSE) for each feature
rmse_prop_damage_category = math.sqrt(mean_squared_error(X_test[0:10]['Property_Damage_Category'], prediction1_df['Property_Damage_Category']))
rmse_dauid = math.sqrt(mean_squared_error(X_test[0:10]['DAUID'], prediction1_df['DAUID']))
rmse_long = math.sqrt(mean_squared_error(X_test[0:10]['Longitude'], prediction1_df['Longitude']))

In [63]:
print(f'RMSE for Property Damage Category: {rmse_prop_damage_category}')
print(f'RMSE for DAUID: {rmse_dauid}')
print(f'RMSE for Longitude: {rmse_long}')

RMSE for Property Damage Category: 120.99556337494815
RMSE for DAUID: 207.32967891146882
RMSE for Longitude: 0.01838507256179243
